In [0]:
# Notebook: 04_generate_orders
# Purpose: Simulate realistic orders data with peak hour patterns for ZapFlow platform
# Layer: Simulation

import random
from datetime import datetime, timedelta
from pyspark.sql.types import *

# ── Configuration ─────────────────────────────────────────────
OUTPUT_PATH = "/Volumes/zapflow_raw/raw_data/raw_files/orders/"
SEED = 42
random.seed(SEED)

# ── Reference data matching previous scripts ───────────────────
CUSTOMER_IDS = [f"CUST_{i:06d}" for i in range(1, 10001)]
STORE_IDS    = [f"STORE_{i:04d}" for i in range(1, 51)]
START_DATE   = datetime(2024, 1, 1)
NUM_DAYS     = 90

# ── Realistic patterns ─────────────────────────────────────────
# Peak hours — lunch and dinner spike
HOUR_WEIGHTS = {
    0: 1,  1: 1,  2: 1,  3: 1,  4: 1,  5: 2,
    6: 3,  7: 5,  8: 6,  9: 5,  10: 4, 11: 6,
    12: 10, 13: 10, 14: 7, 15: 5, 16: 5, 17: 6,
    18: 8, 19: 10, 20: 10, 21: 8, 22: 6, 23: 3
}

ORDER_STATUSES  = ["delivered", "delivered", "delivered", "cancelled", "returned"]
PAYMENT_METHODS = ["UPI", "UPI", "Credit Card", "Debit Card", "Cash on Delivery"]

PRODUCTS_WITH_PRICE = {
    "Tomato": 30,   "Onion": 25,    "Potato": 20,   "Banana": 40,   "Apple": 120,
    "Milk": 60,     "Curd": 45,     "Butter": 55,   "Paneer": 90,   "Cheese": 110,
    "Chips": 30,    "Biscuits": 25, "Namkeen": 35,  "Chocolates": 50, "Cookies": 40,
    "Water": 20,    "Juice": 80,    "Soda": 40,     "Tea": 30,      "Coffee": 150,
    "Shampoo": 180, "Soap": 35,     "Toothpaste": 60, "Facewash": 120, "Sanitizer": 90
}

PRODUCT_NAMES = list(PRODUCTS_WITH_PRICE.keys())

# ── Helper ─────────────────────────────────────────────────────
def weighted_hour():
    hours   = list(HOUR_WEIGHTS.keys())
    weights = list(HOUR_WEIGHTS.values())
    return random.choices(hours, weights=weights, k=1)[0]

def generate_order_items():
    num_items = random.randint(1, 6)
    items     = random.sample(PRODUCT_NAMES, num_items)
    total     = sum(PRODUCTS_WITH_PRICE[item] * random.randint(1, 3) for item in items)
    return ",".join(items), round(total, 2)

def introduce_data_quality_issues(record, counter):
    issue_type = None
    if counter % 50 == 0:
        record["customer_id"] = None
        issue_type = "missing_customer_id"
    elif counter % 80 == 0:
        record["order_amount"] = -1.0
        issue_type = "negative_amount"
    elif counter % 120 == 0:
        record["order_timestamp"] = str(datetime(2099, 1, 1))
        issue_type = "future_timestamp"
    record["_data_quality_issue"] = issue_type
    return record

# ── Generate records ───────────────────────────────────────────
def generate_orders():
    records  = []
    order_id = 1

    for day_offset in range(NUM_DAYS):
        order_date = START_DATE + timedelta(days=day_offset)
        is_weekend = order_date.weekday() >= 5

        # Weekends get more orders
        daily_orders = random.randint(800, 1200) if is_weekend else random.randint(500, 900)

        for _ in range(daily_orders):
            hour      = weighted_hour()
            minute    = random.randint(0, 59)
            second    = random.randint(0, 59)
            timestamp = order_date.replace(hour=hour, minute=minute, second=second)

            items, amount     = generate_order_items()
            status            = random.choice(ORDER_STATUSES)
            delivery_charge   = 0 if amount > 200 else 30
            final_amount      = amount + delivery_charge

            record = {
                "order_id":          f"ORD_{order_id:08d}",
                "customer_id":       random.choice(CUSTOMER_IDS),
                "store_id":          random.choice(STORE_IDS),
                "order_timestamp":   str(timestamp),
                "order_items":       items,
                "order_amount":      amount,
                "delivery_charge":   delivery_charge,
                "final_amount":      final_amount,
                "payment_method":    random.choice(PAYMENT_METHODS),
                "order_status":      status,
                "is_weekend":        is_weekend,
                "_data_quality_issue": None
            }

            record = introduce_data_quality_issues(record, order_id)
            records.append(record)
            order_id += 1

    return records

# ── Main execution ─────────────────────────────────────────────
print("Generating orders data...")
print("This will take a moment — 90 days of orders with peak hour patterns...")

order_records = generate_orders()

schema = StructType([
    StructField("order_id",            StringType(),  False),
    StructField("customer_id",         StringType(),  True),
    StructField("store_id",            StringType(),  True),
    StructField("order_timestamp",     StringType(),  True),
    StructField("order_items",         StringType(),  True),
    StructField("order_amount",        DoubleType(),  True),
    StructField("delivery_charge",     DoubleType(),  True),
    StructField("final_amount",        DoubleType(),  True),
    StructField("payment_method",      StringType(),  True),
    StructField("order_status",        StringType(),  True),
    StructField("is_weekend",          BooleanType(), True),
    StructField("_data_quality_issue", StringType(),  True),
])

df_orders = spark.createDataFrame(order_records, schema=schema)

# ── Save to Volume ─────────────────────────────────────────────
(df_orders
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(OUTPUT_PATH))

print(f"✅ Generated {len(order_records)} order records")
print(f"✅ Saved to {OUTPUT_PATH}")

# ── Sanity checks ──────────────────────────────────────────────
print("\n── Order status distribution ──")
df_orders.groupBy("order_status").count().orderBy("count", ascending=False).show()

print("── Payment method distribution ──")
df_orders.groupBy("payment_method").count().orderBy("count", ascending=False).show()

print("── Weekend vs Weekday orders ──")
df_orders.groupBy("is_weekend").count().show()

print("── Data quality issues ──")
df_orders.groupBy("_data_quality_issue").count().show()

print("── Sample records ──")
df_orders.show(5, truncate=False)